# Notebook 07 — Ensemble Evaluation & Final Comparison
Combines Model A (TF-IDF) and Model C (Chunk+LabelAttn) predictions via weighted average. Produces the final benchmarks table.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import pickle, json
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

from src.config import DATA_DIR, MODEL_A_DIR, MODEL_B_DIR, MODEL_C_DIR, ENSEMBLE_DIR
from src.data import load_label_binarizer
from src.evaluate import full_metrics, tune_global_threshold, head_tail_analysis
from src.models import EnsemblePredictor

mlb   = load_label_binarizer()
vocab = list(mlb.classes_)

print(f'Labels: {len(vocab)}')

## 1. Load All Model Probabilities

In [ ]:
import scipy.sparse as sp

# Ground truth
Y_test  = np.load(DATA_DIR / 'Y_test.npy')
Y_train = np.load(DATA_DIR / 'Y_train.npy')

# Model A: recompute probabilities from classifier
X_test_tfidf = sp.load_npz(DATA_DIR / 'X_test_tfidf.npz')
X_val_tfidf  = sp.load_npz(DATA_DIR / 'X_val_tfidf.npz')
Y_val = np.load(DATA_DIR / 'Y_val.npy')

with open(MODEL_A_DIR / 'clf_sgd.pkl', 'rb') as f:
    clf_a = pickle.load(f)
P_a_test = clf_a.predict_proba(X_test_tfidf)
P_a_val  = clf_a.predict_proba(X_val_tfidf)
with open(MODEL_A_DIR / 'results.json') as f:
    T_a = json.load(f)['test']['threshold']

# Model B
P_b_test = np.load(MODEL_B_DIR / 'P_test.npy')
with open(MODEL_B_DIR / 'test_results.json') as f:
    T_b = json.load(f)['threshold']

# Model C
P_c_test = np.load(MODEL_C_DIR / 'P_test.npy')
P_c_val  = np.load(MODEL_C_DIR / 'P_val.npy')
with open(MODEL_C_DIR / 'test_results.json') as f:
    T_c = json.load(f)['Threshold']

print(f'Model A probs: {P_a_test.shape}  threshold: {T_a}')
print(f'Model B probs: {P_b_test.shape}  threshold: {T_b}')
print(f'Model C probs: {P_c_test.shape}  threshold: {T_c}')

## 2. Tune Ensemble Weights (Model A + Model C)

In [ ]:
# Grid search over weights AND thresholds on validation set
best_w, best_t_ens, best_f1_ens = 0.5, 0.5, 0.0

for w in np.arange(0.0, 1.05, 0.05):
    P_ens_val = w * P_a_val + (1 - w) * P_c_val
    t, f1 = tune_global_threshold(P_ens_val, Y_val)
    if f1 > best_f1_ens:
        best_f1_ens = f1
        best_w = w
        best_t_ens = t

print(f'Best ensemble weight: w={best_w:.2f} (A) + {1-best_w:.2f} (C)')
print(f'Best ensemble threshold: {best_t_ens:.3f}')
print(f'Val micro-F1: {best_f1_ens:.4f}')

# Apply to test
P_ens_test = best_w * P_a_test + (1 - best_w) * P_c_test

## 3. Full Comparison Table

In [ ]:
rows = [
    full_metrics(P_a_test,   Y_test, T_a,        'Model A (TF-IDF + SGD)'),
    full_metrics(P_b_test,   Y_test, T_b,        'Model B (ClinicalBERT)'),
    full_metrics(P_c_test,   Y_test, T_c,        'Model C (Chunk + LabelAttn)'),
    full_metrics(P_ens_test, Y_test, best_t_ens,  f'Ensemble (w={best_w:.2f})'),
]

comparison_df = pd.DataFrame(rows)
print('\n=== Final Model Comparison (Test Set) ===')
print(comparison_df.to_string(index=False))

comparison_df.to_csv(ENSEMBLE_DIR / 'final_comparison_all_models.csv', index=False)
print('\nSaved to', ENSEMBLE_DIR / 'final_comparison_all_models.csv')

## 4. Head / Torso / Tail Breakdown (All Models)

In [ ]:
preds_a   = (P_a_test   >= T_a).astype(int)
preds_b   = (P_b_test   >= T_b).astype(int)
preds_c   = (P_c_test   >= T_c).astype(int)
preds_ens = (P_ens_test >= best_t_ens).astype(int)

f1_a   = f1_score(Y_test, preds_a,   average=None, zero_division=0)
f1_b   = f1_score(Y_test, preds_b,   average=None, zero_division=0)
f1_c   = f1_score(Y_test, preds_c,   average=None, zero_division=0)
f1_ens = f1_score(Y_test, preds_ens, average=None, zero_division=0)

freq = Y_train.sum(0)
label_df = pd.DataFrame({
    'code': vocab, 'freq': freq,
    'f1_A': f1_a, 'f1_B': f1_b, 'f1_C': f1_c, 'f1_Ens': f1_ens,
})

print(f'{"Bucket":25s}  {"n":>5}  {"A":>7}  {"B":>7}  {"C":>7}  {"Ens":>7}')
print('-' * 70)
for lo, hi, name in [(500, 1e9, 'head (>=500)'), (100, 499, 'torso (100-499)'), (0, 99, 'tail (<100)')]:
    s = label_df[(label_df.freq >= lo) & (label_df.freq <= hi)]
    print(f'{name:25s}  {len(s):5d}  {s.f1_A.mean():7.4f}  {s.f1_B.mean():7.4f}  '
          f'{s.f1_C.mean():7.4f}  {s.f1_Ens.mean():7.4f}')

## 5. Visualization — F1 Comparison Across Models

In [ ]:
# Bar chart of key metrics
metrics = ['Micro-F1', 'Macro-F1', 'Micro-Prec', 'Micro-Rec', 'Micro-AUROC']
models  = comparison_df['Model'].tolist()

x = np.arange(len(metrics))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, model_name in enumerate(models):
    row = comparison_df[comparison_df['Model'] == model_name].iloc[0]
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=model_name, alpha=0.85)

ax.set_ylabel('Score')
ax.set_title('Model Comparison (Test Set — Top-50 ICD-10 Codes)')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.legend(loc='lower right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / 'model_comparison_bars.png', dpi=150)
plt.show()

# Scatter: per-label F1 for Model A vs Model C
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(f1_a, f1_c, alpha=0.4, s=15)
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='y=x')
ax.set_xlabel('Model A (TF-IDF) F1'); ax.set_ylabel('Model C (Chunk+Attn) F1')
ax.set_title('Per-Label F1: Model A vs Model C')
ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(ENSEMBLE_DIR / 'a_vs_c_scatter.png', dpi=120)
plt.show()

## 6. Save Ensemble Configuration

In [ ]:
ensemble_config = {
    'weight_model_a': best_w,
    'weight_model_c': 1 - best_w,
    'threshold': best_t_ens,
    'val_micro_f1': best_f1_ens,
    'test_results': rows[3],  # ensemble results
}

with open(ENSEMBLE_DIR / 'ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

np.save(ENSEMBLE_DIR / 'P_ensemble_test.npy', P_ens_test)
print('Ensemble config and predictions saved.')
print(json.dumps(ensemble_config, indent=2))